# 06 — Mapping Layer Demo

This notebook resolves *logical targets* from the IR into *real selectors* using a mapping layer.

In the real system:
- selectors are versioned
- mappings are validated against the app model
- fallbacks and heuristics exist

Here we use a simplified, public‑safe version to illustrate the concept.


In [ ]:
import json
from pathlib import Path


## 6.1 — Load IR Example

We reuse the IR produced in Notebook 02.


In [ ]:
ir_path = Path("../sample-data/ir_examples/login_ir.json")

with open(ir_path, "r") as f:
    ir = json.load(f)

ir


## 6.2 — Define the App Model (TodoMVC Selectors)

This simulates the UI model used by the mapping layer.

These selectors come from the Playwright TodoMVC demo app.


In [ ]:
APP_MODEL = {
    "new_todo_input": {"css": ".new-todo"},
    "todo_item": {"css": ".todo-list li"},
    "toggle_checkbox": {"css": ".toggle"},
    "delete_button": {"css": ".destroy"},
    "todo_count": {"css": ".todo-count"},
    "filter_all": {"css": "a[href='#/']"},
    "filter_active": {"css": "a[href='#/active']"},
    "filter_completed": {"css": "a[href='#/completed']"},
}


## 6.3 — Mapping Resolver

This function resolves logical target names into concrete selectors.

Priority order:
1. CSS
2. ARIA
3. Text
4. URL


In [ ]:
def resolve_target(logical_name: str):
    entry = APP_MODEL.get(logical_name)

    if entry is None:
        return {"error": f"Unknown logical target: {logical_name}"}

    if "css" in entry:
        return {"strategy": "css", "selector": entry["css"]}
    if "aria" in entry:
        return {"strategy": "aria", "selector": entry["aria"]}
    if "text" in entry:
        return {"strategy": "text", "selector": entry["text"]}
    if "url" in entry:
        return {"strategy": "url", "selector": entry["url"]}

    return {"error": f"No valid selector for: {logical_name}"}


## 6.4 — Resolve All Targets in the IR

We apply the mapping layer to each IR step.


In [ ]:
resolved_steps = []

for step in ir["steps"]:
    logical = step["target"]
    resolved = resolve_target(logical)
    resolved_steps.append({
        "action": step["action"],
        "logical_target": logical,
        "resolved": resolved
    })

resolved_steps


## 6.5 — Display Mapping Results

Human‑readable output for inspection.


In [ ]:
for step in resolved_steps:
    print(f"Action: {step['action']}")
    print(f"  Logical:  {step['logical_target']}")
    print(f"  Resolved: {step['resolved']}")
    print()


## 6.6 — Save Mapping Output

This file will be consumed by Notebook 03 for code generation.


In [ ]:
output_path = Path("../sample-data/mapping_output/login_mapping.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(resolved_steps, f, indent=4)

output_path


## Summary

In this notebook we:

- Loaded IR from Notebook 02
- Defined a real APP_MODEL for the Playwright TodoMVC app
- Implemented a simple mapping resolver
- Resolved logical targets into real selectors
- Saved the resolved mapping for Notebook 03

This completes the missing link between IR and real Playwright code generation.
